# Example 1.2.4 / 2: Full Arrays with Default Zero

Companion notebook for **1.2.4 Sparse Parameters** in the AMPL lab Documentation, section "Full Arrays with Default Zero". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Full Arrays with Default Zero

Sometimes every row-column combination is valid, and omission means a coefficient is zero.
Declare the full index domain and a default. Python can still send only the nonzero values;
AMPL supplies zero when another valid entry is referenced:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set ROW;                           # matrix rows
set COL;                           # matrix columns
param coef {ROW, COL} default 0;   # coefficient; omitted entries are zero

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["ROW"] = ["r1", "r2"]
ampl.set["COL"] = ["c1", "c2", "c3"]
ampl.param["coef"] = {("r1", "c1"): 5, ("r1", "c3"): 0,
                      ("r2", "c2"): -2}

In [5]:
# run.py
print(ampl.param["coef"].to_dict())
print(ampl.param["coef"]["r2", "c1"])

{('r1', 'c1'): 5, ('r1', 'c3'): 0, ('r2', 'c2'): -2}
0


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set ROW := r1 r2;
set COL := c1 c2 c3;
param coef := r1 c1 5  r1 c3 0  r2 c2 -2;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display coef;
display coef["r2","c1"];

coef :=
r1 c1    5
r1 c3    0
r2 c2   -2
;

coef['r2','c1'] = 0

